In [ ]:
import meteostat as ms
from meteostat import stations, Point
from datetime import datetime, timedelta
from pathlib import Path
import pandas as pd

# ==========================
# OUTPUT SETUP
# ==========================
OUT = Path("flight_data")
OUT.mkdir(exist_ok=True)

output_file = OUT / "airport_hourly_weather_2025.csv"

# ==========================
# TIME RANGE
# ==========================
start = datetime(2025, 1, 1)
end = datetime(2025, 12, 31, 23, 59)

# ==========================
# AIRPORTS
# ==========================
airports = {
    "TPA": (27.9755, -82.5332),
    "STL": (38.7487, -90.3700),
    "SNA": (33.6757, -117.8678),
    "SMF": (38.6954, -121.5908),
    "SLC": (40.7899, -111.9791),
    "SJC": (37.3639, -121.9289),
    "SFO": (37.6213, -122.3790),
    "SEA": (47.4502, -122.3088),
    "SAT": (29.5337, -98.4698),
    "SAN": (32.7338, -117.1933),
    "RSW": (26.5362, -81.7552),
    "RDU": (35.8801, -78.7880),
    "PIT": (40.4915, -80.2329),
    "PHX": (33.4342, -112.0116),
    "PHL": (39.8744, -75.2424),
    "PDX": (45.5898, -122.5951),
    "ORD": (41.9742, -87.9073),
    "OGG": (20.8987, -156.4305),
    "OAK": (37.7213, -122.2210),
    "MSY": (29.9934, -90.2580),
    "MSP": (44.8848, -93.2223),
    "MIA": (25.7959, -80.2870),
    "MDW": (41.7868, -87.7522),
    "MCO": (28.4312, -81.3081),
    "MCI": (39.2976, -94.7139),
    "LGA": (40.7769, -73.8740),
    "LAX": (33.9416, -118.4085),
    "LAS": (36.0840, -115.1537),
    "JFK": (40.6413, -73.7781),
    "IND": (39.7173, -86.2944),
    "IAH": (29.9902, -95.3368),
    "IAD": (38.9531, -77.4565),
    "HOU": (29.6454, -95.2789),
    "HNL": (21.3187, -157.9225),
    "FLL": (26.0726, -80.1527),
    "EWR": (40.6895, -74.1745),
    "DTW": (42.2162, -83.3554),
    "DFW": (32.8998, -97.0403),
    "DEN": (39.8561, -104.6737),
    "DCA": (38.8512, -77.0402),
    "DAL": (32.8471, -96.8517),
    "CVG": (39.0488, -84.6678),
    "CMH": (39.9980, -82.8919),
    "CLT": (35.2144, -80.9473),
    "CLE": (41.4117, -81.8498),
    "BWI": (39.1754, -76.6684),
    "BOS": (42.3656, -71.0096),
    "BNA": (36.1263, -86.6774),
    "AUS": (30.1975, -97.6664),
    "ATL": (33.6407, -84.4277)
}

# ==========================
# 30-day chunking
# ==========================
def chunk_period(start, end, days=30):
    chunks = []
    cur = start

    while cur < end:
        nxt = min(cur + timedelta(days=days), end)
        chunks.append((cur, nxt))
        cur = nxt

    return chunks

# ==========================
# MAIN LOOP
# ==========================
all_data = []

for code, (lat, lon) in airports.items():
    print(f"Processing {code}...")

    # STEP 1: nearest station (FIXED)
    # STEP 1: nearest station (WORKS in your Meteostat version)
    location = Point(lat, lon)

    nearest = stations.nearby(location).head(1)

    if nearest.empty:
        print(f"⚠️ No station for {code}")
        continue

    station_id = nearest.index[0]

    frames = []

    # STEP 2: hourly data in chunks (FIXED)
    for s, e in chunk_period(start, end, days=30):
        df = ms.hourly(station_id, s, e).fetch()

        if df is not None and not df.empty:
            frames.append(df)

    if not frames:
        print(f"⚠️ No data for {code}")
        continue

    df = pd.concat(frames).reset_index()
    df["airport"] = code

    all_data.append(df)

# ==========================
# CLEAN + FORMAT OUTPUT
# ==========================
if all_data:
    final = pd.concat(all_data, ignore_index=True)

    # --- split datetime index into date + time ---
    final["date"] = final["time"].dt.date
    final["time"] = final["time"].dt.time

    # --- rename columns to plain English ---
    rename_map = {
        "temp": "temperature",
        "dwpt": "dew_point",
        "rhum": "humidity",
        "prcp": "precipitation",
        "snow": "snowfall",
        "wdir": "wind_direction",
        "wspd": "wind_speed",
        "wpgt": "wind_gust",
        "pres": "pressure",
        "tsun": "sunshine_duration"
    }

    final = final.rename(columns=rename_map)

    # --- ensure airport column is clean uppercase codes ---
    final["airport"] = final["airport"].str.upper()

    # (optional) reorder columns nicely
    cols = ["airport", "date", "time"] + [c for c in final.columns if c not in ["airport", "date", "time"]]
    final = final[cols]

    # --- save ---
    final.to_csv(output_file, index=False)

    print(f"Saved to {output_file}")
else:
    print("No data collected.")



Processing TPA...
Processing STL...
Processing SNA...
Processing SMF...
Processing SLC...
Processing SJC...
Processing SFO...
Processing SEA...
Processing SAT...
Processing SAN...
Processing RSW...
Processing RDU...
Processing PIT...
Processing PHX...
Processing PHL...
Processing PDX...
Processing ORD...
Processing OGG...
Processing OAK...
Processing MSY...
Processing MSP...
Processing MIA...
Processing MDW...
Processing MCO...
Processing MCI...
Processing LGA...
Processing LAX...
Processing LAS...
Processing JFK...
Processing IND...
Processing IAH...
Processing IAD...
Processing HOU...
Processing HNL...
Processing FLL...
Processing EWR...
Processing DTW...
Processing DFW...
Processing DEN...
Processing DCA...
Processing DAL...
Processing CVG...
Processing CMH...
Processing CLT...
Processing CLE...
Processing BWI...
Processing BOS...
Processing BNA...
Processing AUS...
Processing ATL...
Saved to flight_data\airport_hourly_weather_2025.csv
